In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# 1. REFERANS TABLOLARINI YÜKLE
# ==========================================
df_ilce_ref = pd.read_excel("data/ilce_koordinat.xlsx")
df_trafik   = pd.read_excel("data/trafik.xlsx")

# ==========================================
# 2. İLÇE ÇÖP VERİSİ (DİNAMİK)
# ==========================================
df_ilceler_raw = pd.read_excel("data/ilceler.xlsx")
df_ilceler_raw.columns = df_ilceler_raw.columns.astype(str).str.strip()
mevcut_yillar = [y for y in ['2021','2022','2023','2024','2025']
                 if y in df_ilceler_raw.columns]

df_ilceler_dinamik = pd.DataFrame({
    'Ilce_Adi':     df_ilceler_raw['İlçe'].astype(str).str.strip().str.title(),
    'Yillik_Tonaj': df_ilceler_raw[mevcut_yillar].mean(axis=1)
})
df_ilceler = pd.merge(df_ilce_ref, df_ilceler_dinamik, on='Ilce_Adi', how='left')
df_ilceler['Yillik_Tonaj'] = df_ilceler['Yillik_Tonaj'].fillna(
    df_ilceler['Yillik_Tonaj'].mean())

# ==========================================
# 3. İSTASYON VERİSİ VE KAPASİTE HESABI
# ==========================================
df_istasyonlar_raw = pd.read_excel("data/istasyonlar.xlsx")
df_istasyonlar = pd.DataFrame({
    'Istasyon_Adi': df_istasyonlar_raw['AKTARMA İSTASYONLARI'].astype(str).str.strip(),
    'Enlem':        df_istasyonlar_raw['LATITUDE'],
    'Boylam':       df_istasyonlar_raw['LONGITUDE'],
    'Alan_m2':      df_istasyonlar_raw['YÜZÖLÇÜMÜ (m2)']
})

Q_TOTAL = 12097.8
df_istasyonlar['Gunluk_Kapasite'] = (
    df_istasyonlar['Alan_m2'] / df_istasyonlar['Alan_m2'].sum()) * Q_TOTAL

asian_station_names = [
    "Küçük Bakkalköy Katı Atık Aktarma İstasyonu",
    "Hekimbaşı Katı Atık Aktama İstasyonu",
    "Aydınlı Katı Atık Aktarma İstasyonu",
    "Şile Katı Atık Aktarma İstasyonu"
]
df_istasyonlar['Yaka'] = np.where(
    df_istasyonlar['Istasyon_Adi'].isin(asian_station_names), 'Asya', 'Avrupa')

# ==========================================
# 4. d_i NORMALİZASYONU
# Ham IBB verisi: sum(d_i) = 17,269 ton/gün
# Sistem kapasitesi (Q_total) = 12,097.8 ton/gün
# Her gün tüm çöp istasyonlara ulaşmalı kısıtı için normalize et.
# ==========================================
d_i_raw = (df_ilceler['Yillik_Tonaj'] / 365.0).to_numpy()
d_i_normalized = d_i_raw * (Q_TOTAL / d_i_raw.sum())
df_ilceler['Gunluk_Tonaj'] = d_i_normalized

print(f"✅ Veriler yüklendi!")
print(f"   İlçe sayısı          : {len(df_ilceler)}")
print(f"   İstasyon sayısı      : {len(df_istasyonlar)}")
print(f"   Ham günlük çöp       : {d_i_raw.sum():.1f} ton/gün")
print(f"   Normalize günlük çöp : {d_i_normalized.sum():.1f} ton/gün")
print(f"   Toplam kapasite      : {Q_TOTAL:.1f} ton/gün")
print(f"\nİstasyon Kapasiteleri:")
for _, row in df_istasyonlar.iterrows():
    kisa = row['Istasyon_Adi'].split()[0]
    print(f"  {kisa:<15} {row['Gunluk_Kapasite']:>7.1f} ton/gün  [{row['Yaka']}]")


In [ ]:
import numpy as np
from scipy.optimize import minimize
from scipy.interpolate import CubicSpline
import folium
import ipywidgets as widgets
from IPython.display import display
import time

# ==========================================
# 1. VERİLERİ 1. HÜCREDEN ÇEKME
# ==========================================
N_DISTRICTS  = len(df_ilceler)
N_STATIONS   = len(df_istasyonlar)
N_ROUTES     = N_DISTRICTS * N_STATIONS   # = 39 × 9 = 351

districts_list   = df_ilceler['Ilce_Adi'].tolist()
d_i              = df_ilceler['Gunluk_Tonaj'].to_numpy()   # normalize edilmiş
asian_districts  = set(df_ilceler[df_ilceler['Yaka'] == 'Asya']['Ilce_Adi'])
districts_coords = list(zip(df_ilceler['Enlem'], df_ilceler['Boylam']))

station_names  = df_istasyonlar['Istasyon_Adi'].tolist()
Q_j            = df_istasyonlar['Gunluk_Kapasite'].to_numpy()
asian_stat_set = set(df_istasyonlar[df_istasyonlar['Yaka'] == 'Asya']['Istasyon_Adi'])
station_coords = list(zip(df_istasyonlar['Enlem'], df_istasyonlar['Boylam']))

D = np.load("data/distance_matrix.npy")
smooth_traffic_curve = CubicSpline(df_trafik["Saat"], df_trafik["Hiz_kmh"])


# ==========================================
# 2. YAKA KONTROLÜ (Boğaz Geçiş Matrisi)
# ==========================================
def build_crossing_mask():
    """True = yanlış yaka (cezalandır), False = aynı yaka (serbest)"""
    mask = np.zeros((N_DISTRICTS, N_STATIONS), dtype=bool)
    for i, d_name in enumerate(districts_list):
        for j, s_name in enumerate(station_names):
            if (d_name in asian_districts) != (s_name in asian_stat_set):
                mask[i, j] = True
    return mask

CROSSING_MASK = build_crossing_mask()


# ==========================================
# 3. PROPOSAL OBJECTIVE FONKSİYONU
# Proposal Denklem (2):
# min Σ_ij D_ij(1 + α·τ(t_ij)²)·x_ij  +  μ·Σ_j[max(0, Σ_i x_ij - Q_j)]²
# ==========================================
def build_penalized_D(CROSSING_PENALTY):
    D_pen = D.copy()
    D_pen[CROSSING_MASK] *= CROSSING_PENALTY
    return D_pen

def objective_function(z, D_penalized, ALPHA, MU):
    x   = z[:N_ROUTES].reshape(N_DISTRICTS, N_STATIONS)
    t   = z[N_ROUTES:].reshape(N_DISTRICTS, N_STATIONS)
    # Trafik yoğunluk indexi τ(t)
    v_t = smooth_traffic_curve(t)
    tau = np.clip(1.0 - (v_t - 27.0) / 9.0, 0.0, 1.0)
    # Taşıma maliyeti
    transport_cost = np.sum(D_penalized * (1.0 + ALPHA * tau**2) * x)
    # Kapasite aşım cezası (Proposal Denklem 2)
    excess         = np.maximum(0.0, np.sum(x, axis=0) - Q_j)
    capacity_pen   = MU * np.sum(excess**2)
    return (transport_cost + capacity_pen) / 10000.0

def equality_constraint(z):
    """Σ_j x_ij = d_i  (tüm çöp her gün taşınmalı)"""
    x = z[:N_ROUTES].reshape(N_DISTRICTS, N_STATIONS)
    return np.sum(x, axis=1) - d_i


# ==========================================
# 4. CROSSING-AWARE PRUNING (Post-Processing)
# Proposal Section F + G'deki budama mantığı:
# Her ilçe için en büyük x_ij'yi al, küçükleri sıfırla.
# Ek kural: seçilen istasyon ilçeyle aynı yakada olmalı.
# ==========================================
def crossing_aware_pruning(x_opt_raw, t_opt_raw, d_i):
    """
    Her ilçeyi en çok çöp gönderdiği istasyona ata (argmax).
    Yanlış yaka seçildiyse → aynı yakada en iyi seçeneğe geç.
    """
    N_D, N_S = x_opt_raw.shape
    x_clean  = np.zeros_like(x_opt_raw)
    t_clean  = np.zeros_like(t_opt_raw)

    for i, d_name in enumerate(districts_list):
        scores    = x_opt_raw[i].copy()
        is_d_asia = d_name in asian_districts

        # Yanlış yaka istasyonlarını sıfırla
        for j, s_name in enumerate(station_names):
            if (s_name in asian_stat_set) != is_d_asia:
                scores[j] = -np.inf

        # Aynı yakada hiç istasyon yoksa (edge case)
        if np.all(np.isinf(scores)):
            scores = x_opt_raw[i].copy()   # crossing olmasa da seçiyoruz

        best_j = int(np.argmax(scores))
        x_clean[i, best_j] = d_i[i]
        t_clean[i, best_j] = t_opt_raw[i, best_j]

    return x_clean, t_clean


# ==========================================
# 5. ANA OPTİMİZASYON FONKSİYONU
# ==========================================
def run_optimization(ALPHA, MU, CROSSING_PENALTY):
    D_penalized = build_penalized_D(CROSSING_PENALTY)

    # Başlangıç noktası: her ilçe → en yakın aynı-yaka istasyon
    x0_matrix = np.zeros((N_DISTRICTS, N_STATIONS))
    for i, d_name in enumerate(districts_list):
        # Yanlış yaka istasyonlarını başlangıçta da engelle
        costs = D_penalized[i].copy()
        x0_matrix[i, int(np.argmin(costs))] = d_i[i]

    z0 = np.concatenate([x0_matrix.flatten(), np.full(N_ROUTES, 14.0)])

    print(f"⚙️  SLSQP çalışıyor [α={ALPHA}, μ={MU:.0f}, boğaz={CROSSING_PENALTY:.0f}x]...")
    t_start = time.time()

    result = minimize(
        objective_function, z0,
        args=(D_penalized, ALPHA, MU),
        method='SLSQP',
        bounds=[(0, None)] * N_ROUTES + [(6.0, 22.0)] * N_ROUTES,
        constraints={'type': 'eq', 'fun': equality_constraint},
        options={'maxiter': 1000, 'ftol': 1e-5, 'eps': 5e-4, 'disp': False}
    )

    elapsed = time.time() - t_start
    print(f"   Süre: {elapsed:.1f} sn  |  Durum: {'✅ Başarılı' if result.success else '⚠️ ' + result.message}")

    x_opt_raw = result.x[:N_ROUTES].reshape(N_DISTRICTS, N_STATIONS)
    t_opt_raw = result.x[N_ROUTES:].reshape(N_DISTRICTS, N_STATIONS)

    # ---- Post-Processing: Crossing-Aware Pruning ----
    x_opt, t_opt = crossing_aware_pruning(x_opt_raw, t_opt_raw, d_i)

    # ---- Sonuç Özeti ----
    station_loads = np.sum(x_opt, axis=0)
    overload      = float(np.sum(np.maximum(0.0, station_loads - Q_j)))
    n_cross       = sum(
        1 for i, d_name in enumerate(districts_list)
        for j in range(N_STATIONS)
        if x_opt[i, j] > 0.1 and CROSSING_MASK[i, j]
    )

    print(f"\n📊 SONUÇLAR")
    print(f"   Aktif Rota       : {int(np.sum(x_opt > 0.1))} (her ilçe 1 rota)")
    print(f"   Kapasite Taşması : {overload:.1f} ton  (ideal: 0)")
    print(f"   Boğaz Geçişi     : {n_cross} rota  (ideal: 0)")
    print(f"\n{'İstasyon':<18} {'Kapasite':>9} {'Yük':>9} {'Doluluk':>9} {'İlçe':>6}")
    print("-" * 55)
    for j in range(N_STATIONS):
        pct    = station_loads[j] / Q_j[j] * 100
        filled = int(min(pct, 100) / 5)
        bar    = '█' * filled + '░' * (20 - filled)
        status = '⚠️' if pct > 110 else '✅'
        kisa   = station_names[j].split()[0]
        ilce_c = int(np.sum(x_opt[:, j] > 0.1))
        print(f"  {status} {kisa:<15} {Q_j[j]:>8.0f}  {station_loads[j]:>8.0f}  {pct:>7.1f}%  {ilce_c:>5}")

    # ---- Harita ----
    m = folium.Map(location=[41.0082, 28.9784], zoom_start=10, tiles="CartoDB dark_matter")

    for idx, coord in enumerate(station_coords):
        pct   = station_loads[idx] / Q_j[idx] * 100
        color = 'red' if pct > 110 else ('orange' if pct > 90 else 'blue')
        kisa  = station_names[idx].split()[0]
        folium.Marker(
            location=coord,
            popup=folium.Popup(
                f"<b>{kisa}</b> [{df_istasyonlar.iloc[idx]['Yaka']}]<br>"
                f"Kapasite: {Q_j[idx]:.0f} ton/gün<br>"
                f"Yük: {station_loads[idx]:.0f} ton ({pct:.1f}%)",
                max_width=220
            ),
            icon=folium.Icon(color=color, icon='trash', prefix='fa')
        ).add_to(m)

    for i, d_name in enumerate(districts_list):
        for j in range(N_STATIONS):
            tonnage = x_opt[i, j]
            if tonnage < 0.5:
                continue
            t_val  = t_opt[i, j]
            hours  = int(t_val)
            mins   = int((t_val - hours) * 60)
            is_cross = CROSSING_MASK[i, j]
            color  = '#ff4444' if is_cross else '#00ff88'  # kırmızı=yanlış yaka!

            # İlçe noktası
            folium.CircleMarker(
                location=districts_coords[i], radius=4,
                color='#ffffff', fill=True,
                fill_color='#aaffcc', fill_opacity=0.9,
                popup=folium.Popup(f"<b>{d_name}</b><br>"
                    f"Çöp: {tonnage:.1f} ton/gün<br>"
                    f"Kalkış: {hours:02d}:{mins:02d}", max_width=200)
            ).add_to(m)

            folium.PolyLine(
                locations=[districts_coords[i], station_coords[j]],
                color=color,
                weight=max(1.5, tonnage / 35.0),
                opacity=0.75,
                tooltip=(
                    f"{'⚠️ BOĞAZ GEÇİŞİ! ' if is_cross else ''}"
                    f"<b>{d_name}</b> → {station_names[j].split()[0]}<br>"
                    f"Mesafe: {D[i,j]:.1f} km | Tonaj: {tonnage:.1f} ton<br>"
                    f"Kalkış: {hours:02d}:{mins:02d}"
                )
            ).add_to(m)

    import os, webbrowser
    map_path = os.path.abspath("data/interaktif_harita.html")
    m.save(map_path)
    print(f"\n🗺️  Harita tarayıcıda açılıyor...")
    webbrowser.open("file://" + map_path)
    display(m)


# ==========================================
# 6. İNTERAKTİF SLIDER'LAR
# ==========================================
interact_ui = widgets.interact_manual(
    run_optimization,
    ALPHA=widgets.FloatSlider(
        value=0.5, min=0.0, max=5.0, step=0.1,
        description='Trafik Ağırlığı (α):',
        style={'description_width': 'initial'}
    ),
    MU=widgets.FloatSlider(
        value=500.0, min=50.0, max=2000.0, step=50.0,
        description='Kapasite Cezası (μ):',
        style={'description_width': 'initial'}
    ),
    CROSSING_PENALTY=widgets.FloatSlider(
        value=100.0, min=10.0, max=500.0, step=10.0,
        description='Boğaz Cezası (×):',
        style={'description_width': 'initial'}
    ),
)
